# 🧱 LEGO AI Training on Google Colab

This notebook allows you to train your YOLOv11 model using Colab's free GPU.

## ⚠️ IMPORTANT: Enable GPU Runtime
Before running, go to **Runtime > Change runtime type** and select **T4 GPU** (or V100/A100 if available).

## Prerequisities
1. Generated a dataset locally using the LEGO Recognition System.
2. Used the local app to sync files to Google Drive (`Lego_data` folder).

## Steps
1. Mount Google Drive.
2. Install Ultralytics.
3. Copy and Unzip Dataset to Colab (Faster I/O).
4. Train Model.
5. Save Model back to Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install Ultralytics (YOLO) & Check GPU
%pip install ultralytics
import torch

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ NO GPU DETECTED! Training will be extremely slow.")
    print("👉 Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# 3. Setup Dataset
import os
import shutil
import yaml

# CONFIGURATION
SET_ID = "42115" # <--- CHANGE THIS TO YOUR SET ID
DRIVE_BASE_DIR = "/content/drive/MyDrive/Lego_data"
ZIP_PATH = os.path.join(DRIVE_BASE_DIR, "datasets", f"{SET_ID}.zip")
LOCAL_DATASET_DIR = f"/content/datasets/{SET_ID}"

if os.path.exists(LOCAL_DATASET_DIR):
    shutil.rmtree(LOCAL_DATASET_DIR)

print(f"Unzipping {ZIP_PATH} to {LOCAL_DATASET_DIR}...")
shutil.unpack_archive(ZIP_PATH, LOCAL_DATASET_DIR)
print("Done!")

# --- FIX PATHS IN DATA.YAML ---
data_yaml_path = os.path.join(LOCAL_DATASET_DIR, "data.yaml")
if os.path.exists(data_yaml_path):
    print(f"Updating paths in {data_yaml_path}...")
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    
    # Overwrite absolute path from local machine with Colab path
    data['path'] = LOCAL_DATASET_DIR
    
    with open(data_yaml_path, 'w') as f:
        yaml.dump(data, f)
    print("✅ data.yaml updated with Colab paths.")
else:
    print(f"⚠️ Warning: data.yaml not found at {data_yaml_path}")

In [ ]:
# 4. Train Model
from ultralytics import YOLO

# Load a model
model = YOLO('yolo11n.pt')  # load a pretrained model (recommended for training)

# Determine device
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

# Train the model
results = model.train(
    data=f"{LOCAL_DATASET_DIR}/data.yaml", 
    epochs=50, 
    imgsz=640, 
    device=device, # Auto-select GPU or CPU
    project='/content/training_results',
    name=f'yolo11_{SET_ID}',
    exist_ok=True # FORCE OVERWRITE: Prevents yolo11_ID-2, -3 folder creation
)

In [ ]:
# 5. Save Model to Drive
import shutil

SOURCE_WEIGHTS = f"/content/training_results/yolo11_{SET_ID}/weights/best.pt"
DEST_DIR = os.path.join(DRIVE_BASE_DIR, "models")
os.makedirs(DEST_DIR, exist_ok=True)
DEST_PATH = os.path.join(DEST_DIR, f"yolo11_{SET_ID}_best.pt")

if os.path.exists(SOURCE_WEIGHTS):
    print(f"Copying model to {DEST_PATH}...")
    shutil.copy(SOURCE_WEIGHTS, DEST_PATH)
    print("✅ Model saved to Google Drive!")
else:
    print(f"❌ ERROR: Model not found at {SOURCE_WEIGHTS}")
    print("Possible reasons:")
    print("1. Training failed completely (check the output of the previous cell).")
    print("2. Training stopped early before generating any weights.")
    
    # Debug: List contents of training results
    print("DEBUG: Listing /content/training_results contents:")
    if os.path.exists("/content/training_results"):
        for root, dirs, files in os.walk("/content/training_results"):
            print(root, files)
    else:
        print("/content/training_results directory does not exist.")